In [ ]:
import shutil
from pathlib import Path
import json

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1 : mini car
2 : medium car
3 : Large car

4 : Compact SUV
5 : Medium SUV
6 : Small Bus
7 : Medium Bus
8 : Large Bus;
9 : Pickup ;
10 : Light DT;
11 : Heavy DT;
12 : Heavy ST;
13 : Light PV;
14 : Heavy FT;
15 : Medium TT;
16 : Mixter Truck ;
17 : Forklift;
18 : Ambulance;
19 : ECV;
20 : Road Roller;
21 : Shovel Loader;
22 : Armored Vehicle;



In [ ]:
!cp /content/drive/MyDrive/Dataset/PIE/Détection_Coco_newClasses/SOC_40classes_coco.tar /content/
!tar -xf /content/SOC_40classes_coco.tar -C /content/

In [ ]:
import json
import os
import glob
from collections import Counter

# --- 1. CONFIGURATION ---
# Remplace par le chemin de ton dossier dans Colab (ex: "/content/dataset/annotations")
PATH_VERS_DATASET = "/content/SOC_40classes_coco/annotations"

# Ton mapping (Ancien_ID: Nouvel_ID)
#mapping_id = {

    #1: 1, 2: 1, 3: 2, 4: 2,
    #5: 3, 6: 3, 7: 3, 8: 4,
    #9: 4, 10: 4, 11: 5, 12: 5,
    #13: 5, 14: 6, 15: 6, 16: 6,
    #17: 6, 18: 7, 19: 7, 20: 7,
    #21: 8, 22: 9, 23: 9, 24: 10,
    #25: 10, 26: 10, 27: 11, 28: 12,
    #29: 12, 30: 13, 31: 14, 32: 14,
    #33: 15, 34: 15, 35: 16, 36: 17,
    #37: 18, 38: 19, 39: 20, 40: 21,
    #41: 22, 42: 22, 43: 22, 44: 22, 45: 22,
    #46: 22, 47: 22, 48: 22, 49: 22, 50: 22
#}

# Tes nouvelles super-catégories
#new_categories = [
    #{"id": 1, "name": "Mini Car", "supercategory": "none"},
    #{"id": 2, "name": "Medium Car", "supercategory": "none"},
    #{"id": 3, "name": "Large Car", "supercategory": "none"},
    #{"id": 4, "name": "Compact SUV", "supercategory": "none"},
    #{"id": 5, "name": "Medium SUV", "supercategory": "none"},
    #{"id": 6, "name": "Small Bus", "supercategory": "none"},
    #{"id": 7, "name": "Medium Bus", "supercategory": "none"},
    #{"id": 8, "name": "Large Bus", "supercategory": "none"},
    #{"id": 9, "name": "Pickup", "supercategory": "none"},
    #{"id": 10, "name": "Light DT", "supercategory": "none"},
    #{"id": 11, "name": "Heavy DT", "supercategory": "none"},
    #{"id": 12, "name": "Heavy ST", "supercategory": "none"},
    #{"id": 13, "name": "Light PV", "supercategory": "none"},
    #{"id": 14, "name": "Heavy FT", "supercategory": "none"},
    #{"id": 15, "name": "Medium TT", "supercategory": "none"},
    #{"id": 16, "name": "Mixter Truck", "supercategory": "none"},
    #{"id": 17, "name": "Forklift", "supercategory": "none"},
    #{"id": 18, "name": "Ambulance", "supercategory": "none"},
    #{"id": 19, "name": "ECV", "supercategory": "none"},
    #{"id": 20, "name": "Road Roller", "supercategory": "none"},
    #{"id": 21, "name": "Shovel Loader", "supercategory": "none"},
    #{"id": 22, "name": "Armored Vehicule", "supercategory": "none"}
#]

mapping_id = {
    1: 1,   # Mini Car → Mini Car
    2: 2,   # Medium Car → Car
    3: 2,   # Large Car → Car
    4: 3,   # Compact SUV → SUV
    5: 3,   # Medium SUV → SUV
    6: 4,   # Small Bus → Small Bus
    7: 5,   # Medium Bus → Medium Bus
    8: 6,   # Large Bus → Large Bus
    9: 7,   # Pickup → Pickup+Light DT
    10: 7,  # Light DT → Pickup+Light DT
    11: 8,  # Heavy DT → Heavy DT
    12: 9,  # Heavy ST → Heavy ST
    13: 10, # Light PV → Light PV
    14: 11, # Heavy FT → Heavy FT
    15: 12, # Medium TT → Medium TT
    16: 13, # Mixer Truck → Mixer Truck
    17: 14, # Forklift → Forklift
    18: 15, # Ambulance → Ambulance
    19: 16, # ECV → ECV
    20: 17, # Road Roller → Construction
    21: 17, # Shovel Loader → Construction
}

new_categories = [
    {"id": 1,  "name": "Mini Car",        "supercategory": "none"},
    {"id": 2,  "name": "Car",             "supercategory": "none"},
    {"id": 3,  "name": "SUV",             "supercategory": "none"},
    {"id": 4,  "name": "Small Bus",       "supercategory": "none"},
    {"id": 5,  "name": "Medium Bus",      "supercategory": "none"},
    {"id": 6,  "name": "Large Bus",       "supercategory": "none"},
    {"id": 7,  "name": "Pickup+Light DT", "supercategory": "none"},
    {"id": 8,  "name": "Heavy DT",        "supercategory": "none"},
    {"id": 9,  "name": "Heavy ST",        "supercategory": "none"},
    {"id": 10, "name": "Light PV",        "supercategory": "none"},
    {"id": 11, "name": "Heavy FT",        "supercategory": "none"},
    {"id": 12, "name": "Medium TT",       "supercategory": "none"},
    {"id": 13, "name": "Mixer Truck",     "supercategory": "none"},
    {"id": 14, "name": "Forklift",        "supercategory": "none"},
    {"id": 15, "name": "Ambulance",       "supercategory": "none"},
    {"id": 16, "name": "ECV",             "supercategory": "none"},
    {"id": 17, "name": "Construction",    "supercategory": "none"},
]


patterns = [
    os.path.join(PATH_VERS_DATASET, "train.json"),
    os.path.join(PATH_VERS_DATASET, "test.json"),
    os.path.join(PATH_VERS_DATASET, "test_*.json")
]

json_files = []
for p in patterns:
    json_files.extend(glob.glob(p))

if not json_files:
    print(f"❌ Aucun fichier JSON trouvé dans {PATH_VERS_DATASET}.")
else:
    for file_path in json_files:
        print(f"\n--- TRAITEMENT DE : {os.path.basename(file_path)} ---")

        with open(file_path, 'r') as f:
            data = json.load(f)

        nb_ann_initial = len(data['annotations'])
        print(f"📊 Nombre d'annotations initiales : {nb_ann_initial}")

        # Liste pour le debug du mapping
        new_ids_stats = []

        # Mise à jour des annotations
        for ann in data['annotations']:
            old_id = ann['category_id']
            # Application du mapping
            new_id = mapping_id.get(old_id, old_id)
            ann['category_id'] = new_id
            new_ids_stats.append(new_id)

            # Optionnel : Correction bbox (tu l'avais enlevé, je le laisse en commentaire)
            # x, y, w, h = ann['bbox']
            # ann['bbox'] = [max(0, x), max(0, y), w, h]

        # Mise à jour de la clé 'categories'
        data['categories'] = new_categories

        # Statistiques de vérification
        compte = Counter(new_ids_stats)
        print(f"✅ Mapping terminé. Nombre de super-catégories uniques trouvées : {len(compte)}")
        print(f"📈 Top 5 nouvelles classes (ID: Nb annotations) : {compte.most_common(5)}")

        # Sauvegarde
        with open(file_path, 'w') as f:
            json.dump(data, f)

        print(f"💾 Fichier sauvegardé avec {len(data['categories'])} définitions de catégories.")

print("\n✨ Tous les fichiers ont été modifiés avec succès !")


--- TRAITEMENT DE : train.json ---
📊 Nombre d'annotations initiales : 68091
✅ Mapping terminé. Nombre de super-catégories uniques trouvées : 17
📈 Top 5 nouvelles classes (ID: Nb annotations) : [(3, 9640), (2, 8472), (7, 8098), (4, 7143), (5, 5643)]
💾 Fichier sauvegardé avec 17 définitions de catégories.

--- TRAITEMENT DE : test.json ---
📊 Nombre d'annotations initiales : 29284
✅ Mapping terminé. Nombre de super-catégories uniques trouvées : 17
📈 Top 5 nouvelles classes (ID: Nb annotations) : [(3, 4085), (2, 3616), (7, 3447), (4, 3142), (5, 2414)]
💾 Fichier sauvegardé avec 17 définitions de catégories.

✨ Tous les fichiers ont été modifiés avec succès !


In [ ]:
# 7. Créer le nouveau .tar
print("\n📦 Création de l'archive SOC_17classes_coco.tar...")
!cd /content && tar -cf SOC_17classes_coco.tar SOC_40classes_coco/

# 8. Copier dans le Drive
print("\n💾 Copie vers Google Drive...")
!cp /content/SOC_17classes_coco.tar /content/drive/MyDrive/Dataset/PIE/Détection_Coco_newClasses/



📦 Création de l'archive SOC_17classes_coco.tar...

💾 Copie vers Google Drive...
